# 05 — Final full-SfM training and RevisitOP evaluation

Only run this notebook after Notebook 04 has written `MyDrive/lightweight-cbir/locked/final_model.json` from SfM validation alone. This notebook retrains that locked design on all SfM-30k training and validation data for its locked best-epoch count, then evaluates it once on held-out Revisited Oxford and Paris.

In [ ]:
# Run this first in every fresh Colab runtime.  It intentionally does not use PYTHONPATH.
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys

# Override this only if you use a fork/private clone URL.
REPO_URL = os.environ.get('CBIR_REPO_URL', 'https://github.com/armin-faraji/LightweightCBIR.git')
REPO_REVISION = os.environ.get('CBIR_REPO_REVISION', 'main')
PROJECT_ROOT = Path('/content/lightweight-cbir')
if not PROJECT_ROOT.is_dir():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', REPO_REVISION], check=True)
os.chdir(PROJECT_ROOT)
required_project_files = (PROJECT_ROOT / 'pyproject.toml', PROJECT_ROOT / 'src' / 'cbir' / '__init__.py', PROJECT_ROOT / 'src' / 'cbir' / 'artifacts.py')
missing_project_files = [str(path.relative_to(PROJECT_ROOT)) for path in required_project_files if not path.is_file()]
if missing_project_files:
    raise RuntimeError('The cloned repository does not contain the Colab-ready project code: ' + ', '.join(missing_project_files) + '. Push the current repository, or set CBIR_REPO_URL/CBIR_REPO_REVISION to the matching commit.')
# A regular reinstall makes the cloned package visible in this current kernel.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', '--force-reinstall', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'h5py', 'scipy', 'PyYAML', 'tqdm', 'matplotlib', 'Pillow'], check=True)

importlib.invalidate_caches()
try:
    import cbir
except ModuleNotFoundError as error:
    raise RuntimeError('The package installation completed but cbir is not visible to this kernel. Restart the runtime once, then rerun this first cell.') from error
print('cbir package:', cbir.__file__)

from cbir.artifacts import create_artifact_run, make_artifact_run_id
from cbir.cloud import mount_colab_drive, runtime_report, write_runtime_report
from cbir.config import config_to_dict, load_project_config
from cbir.utils import stable_hash

PERSISTENT_ROOT = mount_colab_drive() / 'lightweight-cbir'
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('TORCH_HOME', str(PERSISTENT_ROOT / 'torch_hub'))
CONFIG_PATH = Path('configs/colab.yaml')
cfg = load_project_config(CONFIG_PATH)
environment = runtime_report(project_root=PROJECT_ROOT)
config_fingerprint = stable_hash(config_to_dict(cfg))
RUN_ID = make_artifact_run_id(notebook='05', git_sha=environment['git_sha'], config_fingerprint=config_fingerprint)
ARTIFACT_RUN = create_artifact_run(PROJECT_ROOT / 'outputs', '05', run_id=RUN_ID, metadata={'git_sha': environment['git_sha'], 'config_fingerprint': config_fingerprint})
LOCAL_OUTPUT_DIR = ARTIFACT_RUN.local_dir
DRIVE_OUTPUT_ROOT = PERSISTENT_ROOT / 'notebook_outputs'
write_runtime_report(LOCAL_OUTPUT_DIR, project_root=PROJECT_ROOT, extra={'notebook': '05', 'config': str(CONFIG_PATH)})
shutil.copy2(CONFIG_PATH, ARTIFACT_RUN.path_for('config.yaml'))
print('Project:', PROJECT_ROOT)
print('Artifacts:', LOCAL_OUTPUT_DIR)

## Restore the immutable SfM-selected configuration

Notebook 04 selected the architecture, layers, descriptor dimension, and epoch count using SfM validation only. The selected checkpoint is a model-selection record; this notebook trains a new final checkpoint on the merged SfM splits.

In [ ]:
from cbir.utils import read_json

LOCK_PATH = PERSISTENT_ROOT / 'locked' / 'final_model.json'
if not LOCK_PATH.is_file():
    raise FileNotFoundError(f'Notebook 04 has not published a final-model lock: {LOCK_PATH}')
selection_lock = read_json(LOCK_PATH)
if selection_lock.get('selected_on') != 'Manual student decision after SfM-30k validation inspection only':
    raise ValueError('Refusing a final-model lock without SfM-only manual-selection provenance.')
ARTIFACT_RUN.write_json('sfm_selection_lock.json', selection_lock)
print('Locked configuration:', selection_lock['experiment_label'])
print('Final full-SfM training epochs:', selection_lock['best_epoch'])

## Train the locked model on all SfM-30k data

The official SfM training and validation splits are merged only after model selection. Training runs for the 1-based best epoch stored in the lock, without validation or early stopping, while retaining the original 30-epoch cosine learning-rate schedule.

In [ ]:
from cbir.cache import FeatureShardReader, sha256_file
from cbir.cloud import stage_file
from cbir.config import FusionConfig, TrainingConfig, config_to_dict
from cbir.data.sfm import Sfm30kMetadata
from cbir.fusion import build_descriptor_head
from cbir.plotting import SeriesData, plot_series
from cbir.training import HeadTrainer
from cbir.utils import seed_everything
from cbir.workflow import restore_complete_sfm_cache

if cfg.sfm.names_clusters_path is None:
    raise ValueError('configs/colab.yaml must define sfm.names_clusters_path')
LOCAL_SFM_ROOT = cfg.sfm.metadata_path.parent
DRIVE_SFM_ROOT = PERSISTENT_ROOT / 'datasets' / 'sfm30k'
metadata_files = (cfg.sfm.metadata_path, cfg.sfm.names_clusters_path)
LOCAL_SFM_ROOT.mkdir(parents=True, exist_ok=True)
if not all(path.is_file() for path in metadata_files):
    if not all((DRIVE_SFM_ROOT / path.name).is_file() for path in metadata_files):
        raise FileNotFoundError('SfM metadata is absent locally and on Drive; run Notebook 02 first.')
    for path in metadata_files:
        stage_file(DRIVE_SFM_ROOT / path.name, path)
metadata = Sfm30kMetadata.from_official_files(
    cfg.sfm.metadata_path, cfg.sfm.names_clusters_path
)
cache_location = restore_complete_sfm_cache(cfg, metadata)
reader = FeatureShardReader(cache_location.local_dir)
if reader.manifest.fingerprint != selection_lock['cache_fingerprint']:
    raise ValueError('The locked model-selection configuration does not match the restored feature cache.')
# Keep the small frozen cache in RAM so the final fit is not Drive/I/O bound.
reader.max_cached_shards = len(reader.manifest.shards)
warmup_ids = tuple(shard.image_ids[0] for shard in reader.manifest.shards)
_ = reader.fetch(warmup_ids, layer_indices=reader.manifest.layer_indices, local_kind=cfg.fusion.local_kind)
del warmup_ids, _

locked_fusion = dict(selection_lock['fusion_config'])
locked_fusion['layer_indices'] = tuple(int(index) for index in locked_fusion['layer_indices'])
fusion_config = FusionConfig(**locked_fusion)
training_config = TrainingConfig(**selection_lock['training_config'])
FINAL_TRAIN_EPOCHS = int(selection_lock['best_epoch'])
if not 1 <= FINAL_TRAIN_EPOCHS <= training_config.epochs:
    raise ValueError('The locked best epoch is outside the locked training schedule.')
full_sfm_pairs = (*metadata.train_pairs, *metadata.val_pairs)
if not full_sfm_pairs:
    raise RuntimeError('Merged SfM training pairs are empty.')
seed_everything(training_config.seed)
head = build_descriptor_head(fusion_config)
trainer = HeadTrainer(
    head=head,
    reader=reader,
    train_pairs=full_sfm_pairs,
    fusion_config=fusion_config,
    training_config=training_config,
    output_dir=ARTIFACT_RUN.path_for('final_full_sfm_training'),
)
final_history = trainer.fit(
    max_epochs=FINAL_TRAIN_EPOCHS,
    enable_early_stopping=False,
)
if len(final_history.epochs) != FINAL_TRAIN_EPOCHS:
    raise RuntimeError('Final full-SfM training did not complete the locked epoch count.')
FINAL_CHECKPOINT = trainer.save_final_checkpoint(final_history)
final_training = {
    'trained_on': 'merged official SfM-30k train and validation splits',
    'training_pair_count': len(full_sfm_pairs),
    'training_cluster_count': len({pair.cluster_id for pair in full_sfm_pairs}),
    'epochs_completed': len(final_history.epochs),
    'locked_best_epoch': FINAL_TRAIN_EPOCHS,
    'checkpoint_path': str(FINAL_CHECKPOINT),
    'checkpoint_sha256': sha256_file(FINAL_CHECKPOINT),
    'fusion_config': config_to_dict(fusion_config),
    'training_config': config_to_dict(training_config),
    'history': final_history.epochs,
}
ARTIFACT_RUN.write_json('final_full_sfm_training.json', final_training)
figure, _ = plot_series(
    {'Final full-SfM training': SeriesData(
        x=[int(epoch['epoch']) + 1 for epoch in final_history.epochs],
        y=[epoch['loss'] for epoch in final_history.epochs],
    )},
    title='Final full-SfM training loss',
    xlabel='Epoch',
    ylabel='Symmetric InfoNCE loss',
    legend_title='Run',
    save_path=ARTIFACT_RUN.path_for('figures/final_full_sfm_training_loss.png'),
)
print('Final checkpoint:', FINAL_CHECKPOINT)
print('Merged pairs:', len(full_sfm_pairs), '| epochs:', FINAL_TRAIN_EPOCHS)

## Stage Revisited Oxford and Paris

On the first run this downloads and validates the official archives into `/content`, then publishes a validated copy to Drive. On later runtimes it stages the Drive copy back to fast storage. Full JPEG decoding is deliberate: it catches broken downloads before a final benchmark run.

In [ ]:
LOCAL_REVISIT_ROOT = Path('/content/cbir_data/revisitop')
DRIVE_REVISIT_ROOT = PERSISTENT_ROOT / 'datasets' / 'revisitop'
DRIVE_REVISIT_ROOT.mkdir(parents=True, exist_ok=True)
# auto stages a valid Drive dataset when present and downloads only a missing one.
# --publish-root then validates and atomically publishes any new local copy.
stage_command = [
    sys.executable, 'scripts/prepare_revisitop.py',
    '--output-root', str(LOCAL_REVISIT_ROOT),
    '--source-root', str(DRIVE_REVISIT_ROOT),
    '--mode', 'auto',
    '--publish-root', str(DRIVE_REVISIT_ROOT),
    '--repair',
]
subprocess.run(stage_command, cwd=PROJECT_ROOT, check=True)
ARTIFACT_RUN.write_json(
    'revisitop_stage.json',
    read_json(LOCAL_REVISIT_ROOT / 'revisitop_preparation_report.json'),
)
print('RevisitOP staged locally at', LOCAL_REVISIT_ROOT)

## Run the official Medium and Hard protocols

The evaluator uses the newly trained full-SfM checkpoint, extracts full-image database descriptors and one descriptor from each official query bounding-box crop, then applies the official Medium and Hard protocols. It caches descriptor bundles under this run directory, so rerunning the cell reuses matching bundles rather than repeating backbone inference.

In [ ]:
from cbir.plotting import SeriesData, plot_series
from cbir.utils import read_json

BACKBONE_BATCH_SIZE = 8
CHUNK_SIZE = 128
LOCAL_EVALUATION_ROOT = LOCAL_OUTPUT_DIR / 'evaluation'
LOCAL_EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
evaluation_reports = {}
for dataset_name in ('roxford5k', 'rparis6k'):
    command = [
        sys.executable, 'scripts/evaluate_revisitop.py',
        '--config', str(CONFIG_PATH),
        '--checkpoint', str(FINAL_CHECKPOINT),
        '--revisit-root', str(LOCAL_REVISIT_ROOT),
        '--dataset', dataset_name,
        '--output-dir', str(LOCAL_EVALUATION_ROOT),
        '--backbone-batch-size', str(BACKBONE_BATCH_SIZE),
        '--chunk-size', str(CHUNK_SIZE),
    ]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    evaluation_reports[dataset_name] = read_json(
        LOCAL_EVALUATION_ROOT / dataset_name / 'evaluation_report.json'
    )

map_series = {
    'ROxford5k': SeriesData(
        x=[0, 1],
        y=[evaluation_reports['roxford5k']['medium']['map'], evaluation_reports['roxford5k']['hard']['map']],
    ),
    'RParis6k': SeriesData(
        x=[0, 1],
        y=[evaluation_reports['rparis6k']['medium']['map'], evaluation_reports['rparis6k']['hard']['map']],
    ),
}
figure, axis = plot_series(
    map_series,
    title='Final full-SfM model: RevisitOP mAP',
    xlabel='Protocol',
    ylabel='mAP',
    legend_title='Dataset',
)
axis.set_xticks([0, 1], ['Medium', 'Hard'])
ARTIFACT_RUN.save_figure('figures/revisitop_map_medium_hard.png', figure)
ARTIFACT_RUN.write_json('revisitop_summary.json', {
    'sfm_selection_lock': selection_lock,
    'final_full_sfm_training': final_training,
    'backbone_batch_size': BACKBONE_BATCH_SIZE,
    'chunk_size': CHUNK_SIZE,
    'reports': evaluation_reports,
})
print({name: {'medium_map': report['medium']['map'], 'hard_map': report['hard']['map']} for name, report in evaluation_reports.items()})

## Publish notebook outputs to Drive

Run this as the final cell. It validates and publishes the plots, per-query evaluation reports, descriptor-bundle provenance, lock copy, configuration, and runtime details.

In [ ]:
published_output = ARTIFACT_RUN.publish(DRIVE_OUTPUT_ROOT)
print('Validated notebook artifacts published to:', published_output)